<a href="https://colab.research.google.com/github/mhmmdrdhiansyah/data-science-2026/blob/main/pertemuan12_muhammad_ardhiansyah_240401020092.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

nama : Muhammad Ardhiansyah

nim : 240401020092

kelas : IF403

## Langkah 1: Generate & Eksplorasi Dataset Transaksi

Pada tahap ini, kita membuat dataset transaksi belanja sintetis yang berisi 50 transaksi, di mana setiap transaksi terdiri dari 2-5 produk yang dipilih secara acak dari 10 jenis produk. Kita juga menyuntikkan pola pembelian tersembunyi: Roti sering dibeli bersama Selai pada 20 transaksi pertama. Pola ini nantinya akan dideteksi oleh algoritma Apriori sebagai aturan asosiasi yang kuat. Eksplorasi data dilakukan dengan melihat contoh transaksi dan memastikan dataset yang dihasilkan memiliki struktur yang realistis untuk analisis market basket.

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
  
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']
  
# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))
  
# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')
  
print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


## Langkah 2: One-Hot Encoding Transaksi

Algoritma Apriori memerlukan input dalam bentuk tabel one-hot encoding, di mana setiap baris merepresentasikan satu transaksi dan setiap kolom merepresentasikan satu produk. Nilai `True`/`1` berarti produk tersebut dibeli dalam transaksi tersebut, sedangkan `False`/`0` berarti tidak. Kita menggunakan `TransactionEncoder` dari library `mlxtend` untuk mengubah list transaksi menjadi matriks one-hot encoding ini secara otomatis.

In [2]:
from mlxtend.preprocessing import TransactionEncoder
  
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
df.columns = [str(c) for c in df.columns]  # fix numpy 2.x dtype
print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


## Langkah 3: Cari Frequent Itemset dengan Apriori

Apriori adalah algoritma klasik untuk menemukan **frequent itemset** — kombinasi produk yang sering muncul bersamaan dalam transaksi. Algoritma bekerja berdasarkan prinsip: jika sebuah itemset sering muncul, maka semua subset-nya juga harus sering muncul (anti-monotonicity property). Parameter kunci adalah `min_support` — ambang batas frekuensi minimum. Support mengukur seberapa sering sebuah itemset muncul dalam seluruh transaksi.

Pada langkah ini, kita mencoba beberapa nilai `min_support` (0.05, 0.1, 0.2) untuk mengamati bagaimana jumlah itemset yang ditemukan berubah. Semakin rendah min_support, semakin banyak itemset ditemukan (tapi banyak yang tidak signifikan). Semakin tinggi min_support, semakin sedikit tapi lebih kuat. Kita akan menggunakan min_support=0.1 sebagai titik tengah yang seimbang.

In [3]:
from mlxtend.frequent_patterns import apriori
  
for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')
  
# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
# Fix numpy 2.x: pastikan itemsets berisi plain str
freq_items['itemsets'] = freq_items['itemsets'].apply(lambda s: frozenset(str(x) for x in s))
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support                 itemsets
5      0.52       frozenset({Selai})
8      0.46         frozenset({Teh})
3      0.42     frozenset({Mentega})
9      0.36       frozenset({Telur})
1      0.34        frozenset({Keju})
0      0.32        frozenset({Gula})
2      0.32        frozenset({Kopi})
4      0.32        frozenset({Roti})
7      0.32        frozenset({Susu})
36     0.24  frozenset({Teh, Selai})


## Langkah 4: Bentuk & Saring Aturan Asosiasi

Setelah mendapatkan frequent itemset, langkah selanjutnya adalah membentuk **association rules** — aturan berbentuk "Jika beli A, maka beli B" (A → B). Tiga metrik utama untuk menilai kualitas aturan:

- **Support**: Seberapa sering aturan ini muncul dalam seluruh transaksi.
- **Confidence**: Dari semua transaksi yang mengandung A, berapa persen yang juga mengandung B. Mengukur keandalan aturan.
- **Lift**: Rasio confidence terhadap expected confidence. Lift > 1 berarti A dan B muncul bersama lebih sering dari yang diharapkan secara kebetulan — asosiasi positif yang kuat.

Kita menyaring aturan dengan min_confidence=0.5 dan lift > 1 untuk memastikan hanya aturan yang bermakna yang ditampilkan. Aturan dengan lift tertinggi adalah asosiasi terkuat dalam dataset.

In [4]:
from mlxtend.frequent_patterns import association_rules
  
rules = association_rules(freq_items, metric='confidence',
                           min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
  
print(rules[['antecedents', 'consequents',
             'support', 'confidence', 'lift']].head(10))
  
# Interpretasikan dalam sel Markdown:
# Aturan mana yang paling kuat (Lift tertinggi)?
# Apakah masuk akal secara bisnis (mis. Roti -> Selai)?

                    antecedents           consequents  support  confidence  \
8        frozenset({Teh, Keju})    frozenset({Telur})     0.12    0.857143   
15  frozenset({Selai, Mentega})     frozenset({Kopi})     0.10    0.625000   
11      frozenset({Gula, Roti})    frozenset({Selai})     0.10    1.000000   
7           frozenset({Sereal})  frozenset({Mentega})     0.14    0.777778   
9       frozenset({Teh, Telur})     frozenset({Keju})     0.12    0.600000   
14     frozenset({Kopi, Selai})  frozenset({Mentega})     0.10    0.714286   
10     frozenset({Keju, Telur})      frozenset({Teh})     0.12    0.750000   
12     frozenset({Gula, Selai})     frozenset({Roti})     0.10    0.500000   
13   frozenset({Kopi, Mentega})    frozenset({Selai})     0.10    0.714286   
1             frozenset({Roti})    frozenset({Selai})     0.22    0.687500   

        lift  
8   2.380952  
15  1.953125  
11  1.923077  
7   1.851852  
9   1.764706  
14  1.700680  
10  1.630435  
12  1.562500  
13  1.

**Interpretasi Aturan Asosiasi:**

Aturan dengan Lift tertinggi menunjukkan asosiasi terkuat dalam dataset transaksi. Pola Roti → Selai yang disuntikkan di Langkah 1 diharapkan muncul sebagai aturan dengan lift tinggi, karena Roti dan Selai sengaja ditempatkan bersama pada 20 transaksi pertama. Secara bisnis, aturan ini masuk akal — pelanggan yang membeli Roti cenderung juga membeli Selai karena keduanya dikonsumsi bersamaan. Hasil seperti ini dapat dimanfaatkan untuk strategi penempatan produk di toko (cross-merchandising), bundling promo, atau rekomendasi produk di e-commerce.

## Langkah 5: Rekomender Sederhana dengan Content-Based Filtering

Berbeda dengan association rules yang berbasis pola transaksi, **Content-Based Filtering** memberikan rekomendasi berdasarkan kemiripan fitur/atribut produk. Kita membangun katalog produk dengan kategori (Bakery, Dairy, Minuman, Bumbu), lalu menghitung **cosine similarity** antar produk berdasarkan fitur kategori (one-hot encoding). Produk dengan kategori yang sama akan memiliki similarity tinggi dan saling direkomendasikan. Pendekatan ini tidak memerlukan data transaksi, melainkan hanya atribut produk — cocok untuk cold-start problem (produk baru tanpa riwayat transaksi).

In [5]:
from sklearn.metrics.pairwise import cosine_similarity
  
katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
                 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})
  
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)
  
def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()
  
print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


## Langkah 6: Bandingkan Kedua Pendekatan

Pada langkah terakhir, kita membandingkan rekomendasi dari dua pendekatan yang berbeda:

- **Association Rules (Apriori)**: merekomendasikan produk berdasarkan pola pembelian aktual (Roti → Selai karena sering dibeli bersama).
- **Content-Based Filtering**: merekomendasikan produk berdasarkan kemiripan kategori (Roti → Sereal/Mentega karena sama-sama Bakery).

Kedua pendekatan memberikan rekomendasi yang berbeda namun saling melengkapi. Association Rules menangkap pola perilaku pembeli yang nyata, sedangkan Content-Based menangkap kemiripan atribut produk. Di dunia nyata, sistem rekomendasi besar (Netflix, Amazon) biasanya menggunakan pendekatan **hybrid** yang menggabungkan keduanya untuk hasil yang lebih akurat dan komprehensif.

In [6]:
produk_target = 'Roti'
  
# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung produk_target
rules_terkait = rules[rules['antecedents'].apply(
    lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
  
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))
  
# Diskusikan: apakah kedua pendekatan memberi rekomendasi yang konsisten?
# Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?

Rekomendasi dari Association Rules:
           consequents      lift
11  frozenset({Selai})  1.923077
1   frozenset({Selai})  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


## Kesimpulan

Melalui Modul 12 ini, saya mempelajari dua pendekatan utama dalam sistem rekomendasi dan analisis asosiasi data. Pertama, algoritma Apriori yang menemukan frequent itemset dan membentuk association rules dari pola transaksi pembelian, di mana metrik Support, Confidence, dan Lift digunakan untuk menilai kekuatan dan signifikansi aturan. Kedua, Content-Based Filtering yang memberikan rekomendasi berdasarkan kemiripan atribut produk menggunakan cosine similarity, yang sangat berguna untuk menangani masalah cold-start pada produk baru. Dari perbandingan kedua pendekatan, saya memahami bahwa association rules menangkap pola perilaku pembeli nyata (Roti → Selai), sementara content-based filtering menangkap kemiripan kategori (Roti → Sereal/Mentega). Di dunia nyata, sistem rekomendasi modern menggunakan pendekatan hybrid yang menggabungkan keduanya untuk memberikan rekomendasi yang lebih akurat dan personal, seperti yang digunakan oleh Netflix, Amazon, dan Spotify.